# 03 · CheXpert multi-disease fine-tuning (TensorFlow / Keras 3)

This notebook replaces the frozen-backbone model from notebook 02. Compared with 02 it:

* **fine-tunes the whole backbone** (head warm-up first, then every layer except BatchNorm),
* trains on **all frontal CheXpert images** at **320 px** (lateral views are dropped),
* uses a **patient-level** train/validation split, so no patient is in both,
* treats uncertain labels per condition: *U-Ones* for Atelectasis and Edema, *U-Zeros* for the rest (the policies the CheXpert paper found work best),
* reports **per-condition AUROC / sensitivity / specificity** on the official CheXpert validation set, which is radiologist-labelled and used here as the test set,
* picks a **decision threshold per condition** from the internal validation split instead of a hand-set 8%,
* exports a **float32 `.keras` model** with ImageNet normalisation built in, plus a `*_meta.json` file that the HealthAI backend reads (labels, image size, Grad-CAM layer, thresholds, metrics).

## Running on Kaggle
1. New notebook → **File → Import notebook** → upload this file.
2. **Add data** → search "CheXpert" and add a CheXpert-v1.0-small dataset (for example `ashery/chexpert`). The data path is detected automatically.
3. **Settings → Accelerator → GPU T4 x2**, and turn **Internet on** (needed for the ImageNet weights).
4. *Optional:* set `SMOKE_TEST = True` first and run everything. It uses a tiny synthetic dataset and finishes in a few minutes, which checks the pipeline before you spend GPU hours.
5. **Run all**. When it finishes, download `chexpert_<backbone>_<size>.keras` and `chexpert_<backbone>_<size>_meta.json` from the output panel and copy both into `HealthAI-Project-master/models/`.

Rough cost with DenseNet121 at 320 px on 2×T4: about 25–40 min per epoch, so roughly 3–5 h for the default schedule (Kaggle sessions allow 12 h). Data loading is usually the bottleneck.

In [ ]:
import os, json, time, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from sklearn.metrics import roc_auc_score, roc_curve

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

## 1 · Configuration

In [ ]:
SMOKE_TEST = False          # True = tiny synthetic dataset on CPU, to check the pipeline end to end

BACKBONE = "densenet121"    # "densenet121" | "efficientnetv2s" | "convnext_tiny"
IMG_SIZE = 320
BATCH_SIZE = 32             # per GPU
HEAD_EPOCHS = 1             # stage 1: train only the new classifier head
FINETUNE_EPOCHS = 6         # stage 2: train the whole network
HEAD_LR = 1e-3
FINETUNE_LR = 1e-4
WEIGHT_DECAY = 1e-5
EARLY_STOP_PATIENCE = 2
VAL_PATIENT_FRACTION = 0.05 # share of training patients held out for validation / threshold tuning
TRAIN_SUBSET = None         # e.g. 60000 to train on fewer images (faster, a bit less accurate)
TARGET_SENSITIVITY = 0.90   # per-condition thresholds are the highest that reach this sensitivity on validation
SEED = 42
WEIGHTS = "imagenet"

# Optional: path to the old notebook-02 model (.h5), uploaded as a Kaggle dataset, for a before/after comparison
LEGACY_MODEL_PATH = None
LEGACY_IMG_SIZE = 224

DATA_ROOT = None            # folder containing train.csv / valid.csv; None = auto-detect under /kaggle/input
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")

# Must match models/xray_chexpert_labels.json used by the backend
LABELS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
          "Pleural Effusion", "Pneumonia", "Pneumothorax", "No Finding"]
DISEASES = [l for l in LABELS if l != "No Finding"]
COMPETITION_LABELS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
U_ONES = {"Atelectasis", "Edema"}   # uncertain (-1) -> 1 for these, -> 0 for the others

if SMOKE_TEST:
    IMG_SIZE, BATCH_SIZE = 96, 4
    HEAD_EPOCHS, FINETUNE_EPOCHS = 1, 1
    VAL_PATIENT_FRACTION = 0.25
    WEIGHTS = None              # no download needed
    OUTPUT_DIR = OUTPUT_DIR / "smoke_test"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2 · Hardware setup (multi-GPU + mixed precision)

In [ ]:
keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

strategy = tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy()
USE_MIXED_PRECISION = bool(gpus) and not SMOKE_TEST
if USE_MIXED_PRECISION:
    keras.mixed_precision.set_global_policy("mixed_float16")

GLOBAL_BATCH = BATCH_SIZE * strategy.num_replicas_in_sync
print(f"GPUs: {len(gpus)} | replicas: {strategy.num_replicas_in_sync} | global batch: {GLOBAL_BATCH} "
      f"| mixed precision: {USE_MIXED_PRECISION}")
if not gpus and not SMOKE_TEST:
    print("WARNING: no GPU found - full training on CPU will take days. Enable a GPU accelerator.")

## 3 · Locate data (or build the synthetic smoke-test dataset)

In [ ]:
def make_synthetic_chexpert(root, n_train_patients=24, n_valid_patients=8):
    # Writes random-noise JPEGs plus train.csv / valid.csv with the CheXpert layout
    rng = np.random.default_rng(SEED)
    for split, n in (("train", n_train_patients), ("valid", n_valid_patients)):
        rows = []
        for i in range(n):
            pid = f"patient{i + (0 if split == 'train' else 90000):05d}"
            for view, orientation in (("view1_frontal", "Frontal"), ("view2_lateral", "Lateral")):
                rel = f"{split}/{pid}/study1/{view}.jpg"
                path = root / rel
                path.parent.mkdir(parents=True, exist_ok=True)
                img = rng.integers(0, 256, (80, 72, 1), dtype=np.uint8)
                tf.io.write_file(str(path), tf.io.encode_jpeg(img))
                if split == "train":
                    values = rng.choice([np.nan, 0.0, 1.0, -1.0], size=len(LABELS))
                else:  # alternate so every class has positives and negatives
                    values = ((np.arange(len(LABELS)) + i) % 2).astype(float)
                rows.append({"Path": f"CheXpert-v1.0-small/{rel}", "Frontal/Lateral": orientation,
                             **dict(zip(LABELS, values))})
        pd.DataFrame(rows).to_csv(root / f"{split}.csv", index=False)
    return root


def find_chexpert_root():
    if DATA_ROOT:
        return Path(DATA_ROOT)
    # Shallow search only: a recursive glob would walk ~220k image files
    for pattern in ("*/train.csv", "*/*/train.csv", "*/*/*/train.csv", "*/*/*/*/train.csv"):
        for csv in sorted(Path("/kaggle/input").glob(pattern)):
            if (csv.parent / "valid.csv").exists() and (csv.parent / "train").is_dir():
                return csv.parent
    raise FileNotFoundError("CheXpert train.csv/valid.csv not found - add the dataset or set DATA_ROOT")


root = make_synthetic_chexpert(OUTPUT_DIR / "synthetic_chexpert") if SMOKE_TEST else find_chexpert_root()
print("Data root:", root)

In [ ]:
def load_split(csv_name):
    df = pd.read_csv(root / csv_name)
    df = df[df["Frontal/Lateral"] == "Frontal"].copy()
    for lab in LABELS:
        df[lab] = df[lab].fillna(0.0).replace(-1.0, 1.0 if lab in U_ONES else 0.0).astype("float32")
    # CSV paths start with "CheXpert-v1.0-small/"; the dataset folder itself is that directory
    df["filepath"] = df["Path"].map(lambda p: str(root.joinpath(*p.split("/")[1:])))
    df["patient"] = df["Path"].str.extract(r"(patient\d+)", expand=False)
    return df.reset_index(drop=True)


full_train_df = load_split("train.csv")
test_df = load_split("valid.csv")  # official radiologist-labelled set, used only for final evaluation

missing = [p for p in full_train_df["filepath"].head(20) if not os.path.exists(p)]
assert not missing, f"Image paths do not resolve, e.g. {missing[0]} - check DATA_ROOT"

# Patient-level split so no patient appears in both train and validation
patients = full_train_df["patient"].unique()
rng = np.random.default_rng(SEED)
rng.shuffle(patients)
val_patients = set(patients[: max(1, int(len(patients) * VAL_PATIENT_FRACTION))])
val_df = full_train_df[full_train_df["patient"].isin(val_patients)].reset_index(drop=True)
train_df = full_train_df[~full_train_df["patient"].isin(val_patients)].reset_index(drop=True)
if TRAIN_SUBSET and len(train_df) > TRAIN_SUBSET:
    train_df = train_df.sample(TRAIN_SUBSET, random_state=SEED).reset_index(drop=True)

print(f"train: {len(train_df)} images | val: {len(val_df)} | test (official valid): {len(test_df)}")
print("Positive rate per condition:")
pd.DataFrame({name: d[LABELS].mean() for name, d in
              (("train", train_df), ("val", val_df), ("test", test_df))}).round(3)

## 4 · Input pipeline

The model takes **RGB float32 pixels in 0–255**, resized with `tf.image.resize` (bilinear), and does its own normalisation inside the graph. The backend uses exactly the same steps, so training and serving preprocessing can't drift apart.

There is no horizontal flip, because it would move the heart to the wrong side (this matters for Cardiomegaly).

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

augment = keras.Sequential([
    keras.layers.RandomRotation(0.03, fill_mode="constant"),          # about +/-10 degrees
    keras.layers.RandomTranslation(0.05, 0.05, fill_mode="constant"),
    keras.layers.RandomZoom(0.1, fill_mode="constant"),
    keras.layers.RandomContrast(0.1),
    keras.layers.RandomBrightness(0.1),
], name="augment")


def load_image(path, size):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(img, (size, size))  # float32, 0-255


def make_ds(df, training, size=None, scale=1.0):
    size = size or IMG_SIZE
    ds = tf.data.Dataset.from_tensor_slices((df["filepath"].values, df[LABELS].values.astype("float32")))
    if training:
        ds = ds.shuffle(min(len(df), 20000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (load_image(p, size) * scale, y), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(GLOBAL_BATCH, drop_remainder=training)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds = make_ds(train_df, training=True)
val_ds = make_ds(val_df, training=False)
steps_per_epoch = len(train_df) // GLOBAL_BATCH
print("steps per epoch:", steps_per_epoch)

## 5 · Model

The backbone is built on our own input tensor (`input_tensor=`), so its layers sit **flat** inside the final model. That makes Grad-CAM straightforward: the backend reads the last feature-map layer by name (`cam_layer` in the metadata).

In [ ]:
def build_model(weights=WEIGHTS):
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")  # RGB, 0-255
    if BACKBONE == "densenet121":
        x = keras.layers.Rescaling(1.0 / 255, name="scale_0_1")(inputs)
        x = keras.layers.Normalization(mean=[0.485, 0.456, 0.406],
                                       variance=[0.229 ** 2, 0.224 ** 2, 0.225 ** 2],
                                       name="imagenet_norm")(x)
        base = keras.applications.DenseNet121(include_top=False, weights=weights, input_tensor=x)
    elif BACKBONE == "efficientnetv2s":  # preprocessing is built into the Keras application
        base = keras.applications.EfficientNetV2S(include_top=False, weights=weights, input_tensor=inputs)
    elif BACKBONE == "convnext_tiny":
        base = keras.applications.ConvNeXtTiny(include_top=False, weights=weights, input_tensor=inputs)
    else:
        raise ValueError(f"Unknown BACKBONE {BACKBONE}")

    features = base.output
    try:
        cam_layer = features._keras_history.operation.name
    except AttributeError:
        cam_layer = base.layers[-1].name

    x = keras.layers.GlobalAveragePooling2D(name="gap")(features)
    x = keras.layers.Dropout(0.3, name="head_dropout")(x)
    outputs = keras.layers.Dense(len(LABELS), activation="sigmoid", dtype="float32", name="predictions")(x)
    model = keras.Model(inputs, outputs, name=f"chexpert_{BACKBONE}")

    head = {"gap", "head_dropout", "predictions", "image"}
    backbone_layers = [l for l in model.layers if l.name not in head]
    return model, backbone_layers, cam_layer


def set_backbone_trainable(backbone_layers, trainable):
    for layer in backbone_layers:
        # BatchNorm stays frozen (inference mode) during fine-tuning - the usual Keras recipe
        layer.trainable = trainable and not isinstance(layer, keras.layers.BatchNormalization)


def compile_model(model, schedule):
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=schedule, weight_decay=WEIGHT_DECAY),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.AUC(multi_label=True, num_labels=len(LABELS), name="auc")],
    )


with strategy.scope():
    model, backbone_layers, CAM_LAYER = build_model()

cam_shape = model.get_layer(CAM_LAYER).output.shape
assert len(cam_shape) == 4, f"Grad-CAM layer {CAM_LAYER} is not a feature map: {cam_shape}"
print(f"{model.name}: {model.count_params():,} params | Grad-CAM layer: {CAM_LAYER} {cam_shape}")

## 6 · Stage 1: warm up the new head (backbone frozen)

In [ ]:
history = {}
t0 = time.time()
with strategy.scope():
    set_backbone_trainable(backbone_layers, False)
    compile_model(model, HEAD_LR)
history["head"] = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS).history
print(f"Stage 1 done in {(time.time() - t0) / 60:.1f} min")

## 7 · Stage 2: fine-tune the whole network

In [ ]:
ft_steps = max(1, steps_per_epoch * FINETUNE_EPOCHS)
warmup = min(500, ft_steps // 10)
schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=FINETUNE_LR / 10, decay_steps=ft_steps - warmup,
    warmup_target=FINETUNE_LR, warmup_steps=warmup)

callbacks = [
    keras.callbacks.ModelCheckpoint(str(OUTPUT_DIR / "checkpoint_best.keras"),
                                    monitor="val_auc", mode="max", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=EARLY_STOP_PATIENCE,
                                  restore_best_weights=True),
    keras.callbacks.CSVLogger(str(OUTPUT_DIR / "finetune_log.csv")),
]

t0 = time.time()
with strategy.scope():
    set_backbone_trainable(backbone_layers, True)
    compile_model(model, schedule)
history["finetune"] = model.fit(train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS,
                                callbacks=callbacks).history
print(f"Stage 2 done in {(time.time() - t0) / 60:.1f} min | best val AUROC "
      f"{max(history['finetune']['val_auc']):.4f}")

## 8 · Evaluation and per-condition thresholds

Thresholds are tuned on the **internal validation split** and then applied unchanged to the **official CheXpert validation set**, so the test numbers are honest.

The test set is small (about 200 frontal images, with only a handful of Pneumonia and Pneumothorax positives), so those per-class numbers are noisy.

In [ ]:
def labels_of(df):
    return df[LABELS].values.astype("float32")


def pick_thresholds(y, p, target_sensitivity):
    sens_t, youden_t = {}, {}
    for i, lab in enumerate(LABELS):
        yi, pi = y[:, i], p[:, i]
        if not 0 < yi.sum() < len(yi):
            sens_t[lab] = youden_t[lab] = 0.5
            continue
        fpr, tpr, thr = roc_curve(yi, pi)
        thr = np.clip(thr, 0.0, 1.0)              # sklearn's first threshold is +inf
        sens_t[lab] = float(thr[np.argmax(tpr >= target_sensitivity)])  # highest threshold reaching target
        youden_t[lab] = float(thr[np.argmax(tpr - fpr)])
    return sens_t, youden_t


def per_class_metrics(y, p, thresholds=None):
    rows = {}
    for i, lab in enumerate(LABELS):
        yi, pi = y[:, i].astype(bool), p[:, i]
        row = {"positives": int(yi.sum()), "n": int(len(yi)),
               "auroc": float(roc_auc_score(yi, pi)) if 0 < yi.sum() < len(yi) else None}
        if thresholds:
            pred = pi >= thresholds[lab]
            tp, fn = int((pred & yi).sum()), int((~pred & yi).sum())
            tn, fp = int((~pred & ~yi).sum()), int((pred & ~yi).sum())
            row.update(threshold=thresholds[lab],
                       sensitivity=tp / (tp + fn) if tp + fn else None,
                       specificity=tn / (tn + fp) if tn + fp else None)
        rows[lab] = row
    return rows


def mean_auroc(metrics, labels):
    vals = [metrics[l]["auroc"] for l in labels if metrics[l]["auroc"] is not None]
    return float(np.mean(vals)) if vals else None


y_val, y_test = labels_of(val_df), labels_of(test_df)
p_val = model.predict(val_ds, verbose=0)
p_test = model.predict(make_ds(test_df, training=False), verbose=0)

THRESHOLDS, YOUDEN_THRESHOLDS = pick_thresholds(y_val, p_val, TARGET_SENSITIVITY)
val_metrics = per_class_metrics(y_val, p_val, THRESHOLDS)
test_metrics = per_class_metrics(y_test, p_test, THRESHOLDS)

summary = {
    "val_mean_auroc": mean_auroc(val_metrics, LABELS),
    "test_mean_auroc": mean_auroc(test_metrics, LABELS),
    "test_mean_auroc_competition5": mean_auroc(test_metrics, COMPETITION_LABELS),
}
print(json.dumps(summary, indent=2))
pd.DataFrame(test_metrics).T

### Optional: compare with the old notebook-02 model

In [ ]:
legacy_summary = None
if LEGACY_MODEL_PATH and Path(LEGACY_MODEL_PATH).exists():
    legacy = keras.models.load_model(LEGACY_MODEL_PATH, compile=False)
    p_legacy = legacy.predict(make_ds(test_df, training=False, size=LEGACY_IMG_SIZE, scale=1 / 255.0), verbose=0)
    legacy_metrics = per_class_metrics(y_test, p_legacy)
    legacy_summary = {"test_mean_auroc": mean_auroc(legacy_metrics, LABELS),
                      "test_mean_auroc_competition5": mean_auroc(legacy_metrics, COMPETITION_LABELS)}
    display(pd.DataFrame({"old AUROC": {l: m["auroc"] for l, m in legacy_metrics.items()},
                          "new AUROC": {l: m["auroc"] for l, m in test_metrics.items()}}))
    print("old:", legacy_summary, "\nnew:", summary)
else:
    print("LEGACY_MODEL_PATH not set - skipping comparison")

## 9 · Export for the backend

The model is rebuilt in **float32** and the trained weights are copied in. A mixed-precision model would run slowly, or fail, on the backend's CPU.

In [ ]:
keras.mixed_precision.set_global_policy("float32")
export_model, _, export_cam_layer = build_model(weights=None)
export_model.set_weights(model.get_weights())

sample = next(iter(make_ds(test_df.head(GLOBAL_BATCH), training=False)))[0]
max_diff = float(np.max(np.abs(export_model.predict(sample, verbose=0) - model.predict(sample, verbose=0))))
assert max_diff < 0.05, f"float32 export differs from trained model by {max_diff}"

MODEL_NAME = f"chexpert_{BACKBONE}_{IMG_SIZE}"
model_path = OUTPUT_DIR / f"{MODEL_NAME}.keras"
meta_path = OUTPUT_DIR / f"{MODEL_NAME}_meta.json"
export_model.save(model_path)

meta = {
    "model_file": model_path.name,
    "name": f"CheXpert-{BACKBONE}-{IMG_SIZE}-finetuned",
    "backbone": BACKBONE,
    "labels": LABELS,
    "img_size": IMG_SIZE,
    "preprocess": "rgb_0_255",  # RGB float32 0-255, tf.image.resize bilinear; normalisation is inside the model
    "cam_layer": export_cam_layer,
    "thresholds": THRESHOLDS,
    "threshold_policy": f"highest threshold with sensitivity >= {TARGET_SENSITIVITY:.0%} on internal validation",
    "youden_thresholds": YOUDEN_THRESHOLDS,
    "uncertainty_policy": {l: ("U-Ones" if l in U_ONES else "U-Zeros") for l in LABELS},
    "metrics": {"summary": summary, "test": test_metrics, "val": val_metrics, "legacy_test": legacy_summary},
    "training": {"head_epochs": HEAD_EPOCHS, "finetune_epochs_run": len(history["finetune"]["loss"]),
                 "train_images": len(train_df), "val_images": len(val_df), "test_images": len(test_df),
                 "global_batch": GLOBAL_BATCH, "smoke_test": SMOKE_TEST},
    "versions": {"tensorflow": tf.__version__, "keras": keras.__version__},
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
}
meta_path.write_text(json.dumps(meta, indent=2))
print(f"Saved {model_path} ({model_path.stat().st_size / 1e6:.1f} MB) and {meta_path.name} "
      f"| float32 max diff {max_diff:.2e}")

## 10 · Sanity check: reload the export and draw a Grad-CAM

In [ ]:
import matplotlib.pyplot as plt

reloaded = keras.models.load_model(model_path, compile=False)
grad_model = keras.Model(reloaded.inputs, [reloaded.get_layer(meta["cam_layer"]).output, reloaded.output])

row = test_df.iloc[0]
img = load_image(row["filepath"], IMG_SIZE)[None]
with tf.GradientTape() as tape:
    conv, preds = grad_model(img)
    disease_scores = tf.gather(preds[0], [LABELS.index(d) for d in DISEASES])
    class_idx = LABELS.index(DISEASES[int(tf.argmax(disease_scores))])
    score = preds[:, class_idx]
grads = tape.gradient(score, conv)
cam = tf.nn.relu(tf.reduce_sum(conv[0] * tf.reduce_mean(grads, axis=(0, 1, 2)), axis=-1)).numpy()
cam = cam / cam.max() if cam.max() > 0 else cam

fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(img[0].numpy().astype("uint8")); ax[0].set_title("input")
ax[1].imshow(img[0].numpy().astype("uint8"))
ax[1].imshow(tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE))[..., 0], cmap="jet", alpha=0.45)
ax[1].set_title(f"Grad-CAM: {LABELS[class_idx]} ({float(preds[0, class_idx]):.1%})")
for a in ax: a.axis("off")
plt.show()

print("Copy these two files into HealthAI-Project-master/models/:")
print(" ", model_path.name, "\n ", meta_path.name)